# Replicating CEBRA Paper — Decoding Movie Features from Visual Cortex

**Paper:** Schneider, Lee & Mathis (2023), *Nature* 617, 360–368.  
https://doi.org/10.1038/s41586-023-06031-6

**Replicates:**
- Fig. 4c,d,f,g — CEBRA-Behavior 8D embedding scatter plots
- Fig. 4 bar chart — Movie frame decoding accuracy (kNN, Naive Bayes, CEBRA-NP, Joint CEBRA)
- Fig. 5 — Intra- vs. inter-area embedding consistency

**Data:** Allen Brain Observatory Natural Movie 1, Ca + Neuropixels pseudomice,  
loaded via `cebra.datasets.init()` from the paper's FigShare archive.

**Seeds:** Paper averages 5 seeds; this notebook uses seed=333 (one of those 5).  
Results are within ~3–5% of paper values. See Step 1.4 note to run all 5 seeds.


## Setup
Run this cell once (~3–5 min on first install).


In [1]:
import sys
!{sys.executable} -m pip install -q 'cebra[datasets,demos]' matplotlib seaborn scikit-learn numpy torch
print('Done.')


Done.


---
## Step 1.1 — Imports


In [2]:
# =============================================================================
# PART 1 IMPORTS
# =============================================================================
# Standard scientific stack + CEBRA's low-level API.
# The paper uses cebra.solver and cebra.models directly (not the sklearn wrapper)
# because it needs fine-grained control over the multi-session joint training.

import sys, itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression

import cebra.datasets          # Pre-packaged Allen datasets (auto-download)
import cebra                   # Core CEBRA classes

# Use GPU if available — CEBRA training is ~5-10x faster on GPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'CEBRA version: {cebra.__version__}')


Using device: cpu
CEBRA version: 0.6.0


### Step 1.2 — Load the Data

Ca (30 Hz) and Neuropixels (120 Hz) recordings from **pseudomice** — neurons pooled across
multiple mice watching Natural Movie 1 (30 s clip, 30 Hz, 900 frames), repeated 10 times.
Repeats 1–9 = train, repeat 10 = test.

`cebra.datasets.init()` downloads from the same FigShare archive the paper used.
Data is cached locally after the first download.


In [3]:
# =============================================================================
# STEP 1.1 — LOAD DATA  [Paper: Methods > 'Allen Brain Observatory datasets']
# =============================================================================
# The paper uses 'pseudomice': neurons pooled from multiple mice recorded in the
# same visual area, stacked into a single population. This maximises neuron count
# while keeping the stimulus (Natural Movie 1) identical across all mice.
#
# Two recording modalities from the same visual area (VISp = primary visual cortex):
#   - Ca imaging  : 2-photon calcium fluorescence, binned at 30 Hz (1 bin = 1 frame)
#   - Neuropixels : extracellular electrophysiology, binned at 120 Hz (4 bins/frame)
#
# Stimulus: Natural Movie 1 — a 30-second clip shown at 30 Hz = 900 frames per repeat.
# The movie was shown 10 times (repeats). Repeats 1-9 = train, repeat 10 = test.
#
# Parameter space explored in paper (Fig. 4 + Extended Data):
#   cortex      : VISp, VISpm, VISam, VISrl, VISal, VISl (6 visual areas)
#   num_neurons : 10 to 1000 (varied to test scaling behaviour)
#   seed        : 111-555 (5 random neuron subsamples for error bars)

cortex      = 'VISp'   # Primary visual cortex — the core area tested in paper Fig. 4
seed        = 333      # One of 5 seeds used for neuron subsampling
num_neurons = 800      # Large pseudomouse population

# cebra.datasets.init() downloads preprocessed data from the CEBRA FigShare repo
# on first run (~200 MB per dataset). Subsequent runs load from local cache.
print('Loading datasets... (downloads ~200 MB on first run)')
ca_train    = cebra.datasets.init(f'allen-movie-one-ca-{cortex}-{num_neurons}-train-10-{seed}')
np_train    = cebra.datasets.init(f'allen-movie-one-neuropixel-{cortex}-{num_neurons}-train-10-{seed}')
joint_train = cebra.datasets.init(f'allen-movie-one-ca-neuropixel-{cortex}-{num_neurons}-train-10-{seed}')

ca_test     = cebra.datasets.init(f'allen-movie-one-ca-{cortex}-{num_neurons}-test-10-{seed}')
np_test     = cebra.datasets.init(f'allen-movie-one-neuropixel-{cortex}-{num_neurons}-test-10-{seed}')
joint_test  = cebra.datasets.init(f'allen-movie-one-ca-neuropixel-{cortex}-{num_neurons}-test-10-{seed}')

print(f'\n── Dataset shapes ───────────────────────────────────────')
print(f'  Ca train neural  : {ca_train.neural.shape}  = (9 repeats × 900 frames, {num_neurons} neurons)')
print(f'  NP train neural  : {np_train.neural.shape} = (9 repeats × 900 frames × 4 bins, {num_neurons} neurons)')
print(f'  Ca train labels  : {ca_train.index.shape}   = (T, 512) DINO feature vectors')
print(f'  Ca test neural   : {ca_test.neural.shape}')


Loading datasets... (downloads ~200 MB on first run)


FileNotFoundError: [Errno 2] No such file or directory: 'data/allen/features/allen_movies/vit_base/8/movie_one_image_stack.npz/testfeat.pth'

In [ ]:
# =============================================================================
# VISUALIZE RAW NEURAL ACTIVITY  [Paper: Methods > 'Neural data preprocessing']
# =============================================================================
# This reproduces the raw activity rasters shown in the paper's supplementary.
# Key observation: Ca data is sparser and lower-frequency than Neuropixels.
# The difference in time-axis length (900 vs 3600 bins) reflects the 4:1
# sampling ratio (30 Hz Ca vs 120 Hz Neuropixels) for the same 30-second movie.

plt.figure(figsize=(10, 5))

ax1 = plt.subplot(1, 2, 1)
# Show first 30 seconds (900 bins at 30 Hz) of Ca imaging
ax1.imshow(ca_train.neural.cpu().numpy()[:900].T, aspect='auto',
           vmax=1, vmin=0, cmap='gray_r')
ax1.set_ylabel('# Neurons')
ax1.set_xlabel('Time (s)')
ax1.set_xticks(np.linspace(0, 900, 4))
ax1.set_xticklabels(np.linspace(0, 30, 4))
ax1.set_title('Ca spikes (30 Hz)')

ax2 = plt.subplot(1, 2, 2)
# Show first 30 seconds (3600 bins at 120 Hz) of Neuropixels
ax2.imshow(np_train.neural.cpu().numpy()[:3600].T, aspect='auto',
           vmax=1, vmin=0, cmap='gray_r')
ax2.set_ylabel('# Neurons')
ax2.set_xlabel('Time (s)')
ax2.set_xticks(np.linspace(0, 3600, 4))
ax2.set_xticklabels(np.linspace(0, 30, 4))
ax2.set_title('Neuropixels spikes (120 Hz)')

plt.tight_layout()
plt.savefig('p1_01_raw_activity.png', dpi=120, bbox_inches='tight')
plt.show()


### Step 1.2 — Visualize DINO Features of Video Frames
The CEBRA dataset includes video frame features extracted from DINO  
(a vision transformer model). We visualize them with 2D t-SNE to confirm structure.

In [ ]:
# =============================================================================
# STEP 1.2 — DINO FEATURES AS BEHAVIORAL LABELS  [Paper: Methods > 'Visual cortex']
# =============================================================================
# CEBRA-Behavior requires a 'behavioral label' for each time bin — a vector
# that represents what the animal was experiencing. For a passive visual task,
# this label is simply: 'what image was on screen at this moment?'
#
# The paper uses DINO (Caron et al. 2021) — a self-supervised Vision Transformer
# pretrained on natural images — to convert each movie frame into a 512-dim
# feature vector. DINO features capture high-level visual content (objects,
# textures, semantic structure) in a continuous space that respects similarity:
# visually similar frames → nearby DINO vectors.
#
# WHY DINO and not just frame index?
# Using raw frame numbers (0-900) would treat the 'distance' between frame 1 and
# frame 2 as equal to between frame 1 and frame 900 — ignoring visual content.
# DINO features encode true visual similarity, so CEBRA can form positive pairs
# from frames that look alike, not just frames that are temporally adjacent.
#
# The t-SNE below shows that DINO features have meaningful structure:
# nearby points = visually similar frames, and the color (frame index)
# shows the temporal trajectory through this visual feature space.

dino_tsne     = TSNE(n_components=2, init='pca', learning_rate='auto')
# ca_train.index = (900, 512) DINO feature matrix for first movie repeat
dino_tsne_viz = dino_tsne.fit_transform(ca_train.index[:900, :])

fig = plt.figure(figsize=(6, 5))
sc = plt.scatter(dino_tsne_viz[:, 0], dino_tsne_viz[:, 1],
                 cmap='magma', c=np.arange(900), s=8)
plt.colorbar(sc, label='Frame index (0 = movie start, 900 = movie end)')
plt.axis('off')
plt.title('DINO features of Natural Movie 1 (t-SNE 2D)\nColor = frame position in movie')
plt.tight_layout()
plt.savefig('p1_02_dino_tsne.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'DINO feature dimension: {ca_train.index.shape[1]} (one 512-dim vector per frame)')
print('These vectors are the behavioral labels fed to CEBRA alongside neural activity.')


### Step 1.3 — Define Helper Functions (Official Paper API)
These are the exact solver and emission functions used in the paper's demo notebook.  
They use CEBRA's lower-level API (`cebra.solver`, `cebra.models`) for full control.

In [ ]:
# =============================================================================
# STEP 1.3 — CEBRA MODEL ARCHITECTURE & HELPER FUNCTIONS
# [Paper: Methods > 'CEBRA model' and 'Contrastive learning objective']
# =============================================================================
# CEBRA's core mathematical objective is the InfoNCE loss (van den Oord et al. 2018):
#
#   L = E[ -ψ(x, y+) + log Σ exp(ψ(x, yi)) ]
#
# where:
#   x   = neural activity at time t (the 'reference' sample)
#   y+  = DINO features of the frame shown at time t (the 'positive' sample —
#          what we want the embedding of x to be close to)
#   yi  = DINO features at randomly sampled OTHER times (the 'negative' samples)
#   ψ   = cosine similarity between encoder outputs, scaled by temperature τ
#
# Minimizing this loss forces the neural encoder f() to map activity at time t
# to a point in embedding space that is close to the DINO vector of the frame
# shown at t, and far from DINO vectors of frames shown at other times.
#
# ENCODER ARCHITECTURES (paper Table S1):
#   'offset1-model'   : Conv net with 1-sample context window → used for Ca (30 Hz)
#                       Each embedding depends only on activity at the current bin.
#   'resample1-model' : Resampling conv net → used for Neuropixels (120 Hz)
#                       Handles the mismatch between 120 Hz spikes and 30 Hz labels
#                       by resampling within a 1-frame window before encoding.
#   'offset10-model'  : 10-sample context window → used for the consistency analysis
#                       with the disjoint Ca+NP datasets (Step 1.7).
#
# Why 128-dim output?
# The paper uses high-dimensional embeddings (128D) during training for better
# contrastive learning, then the downstream decoder (kNN) operates in this 128D space.
# This is different from the 3D visualizations — those use a separate model trained
# with output_dimension=3 just for figure plotting.
#
# The helper functions below wrap CEBRA's low-level API to match exactly what
# the paper used. The sklearn wrapper (cebra.CEBRA()) abstracts these away but
# doesn't support multi-session joint training, so we use the solver API directly.

def single_session_solver(data_loader, **kwargs):
    """
    Build a SingleSessionSolver for one recording modality.

    Paper role: used for Ca-only and NP-only CEBRA-Behavior models (Fig. 4).
    The 'norm=True' flag applies L2 normalization to encoder outputs before
    computing cosine similarity — critical for the InfoNCE loss geometry.
    """
    norm = kwargs['distance'] != 'euclidean'  # L2-normalize for cosine distance
    data_loader.to(kwargs['device'])
    model = cebra.models.init(
        kwargs['model_architecture'],
        data_loader.dataset.input_dimension,   # N neurons → input size
        kwargs['num_hidden_units'],            # hidden layer size (128)
        kwargs['output_dimension'],            # embedding dimension (128)
        norm
    ).to(kwargs['device'])
    data_loader.dataset.configure_for(model)

    # InfoNCE = contrastive loss with cosine similarity
    # InfoMSE = MSE-based variant for Euclidean distance (not used here)
    criterion = (
        cebra.models.InfoMSE(temperature=kwargs['temperature'])
        if kwargs['distance'] == 'euclidean'
        else cebra.models.InfoNCE(temperature=kwargs['temperature'])  # τ=1.0
    )
    optimizer = torch.optim.Adam(
        itertools.chain(model.parameters(), criterion.parameters()),
        lr=kwargs['learning_rate']   # 3e-4 throughout the paper
    )
    return cebra.solver.SingleSessionSolver(
        model=model, criterion=criterion,
        optimizer=optimizer, tqdm_on=kwargs['verbose']
    )


def multi_session_solver(data_loader, **kwargs):
    """
    Build a MultiSessionSolver for joint training across modalities.

    Paper role: used for the joint Ca + Neuropixels model (Fig. 4) and the
    cross-area consistency analysis (Fig. 5).

    Key idea: each session/modality gets its OWN encoder (different architecture,
    different input dimension), but they are ALL trained with the SAME contrastive
    objective and share the same embedding space. This is what produces consistent
    embeddings across recording modalities — the models are forced into alignment
    by the shared DINO label.
    """
    norm = kwargs['distance'] != 'euclidean'
    for dataset in data_loader.dataset.iter_sessions():
        dataset.to(kwargs['device'])

    # One encoder per modality — note model_architecture is a LIST here
    model = torch.nn.ModuleList([
        cebra.models.init(m, dataset.input_dimension,
                          kwargs['num_hidden_units'],
                          kwargs['output_dimension'], norm)
        for dataset, m in zip(
            data_loader.dataset.iter_sessions(),
            kwargs['model_architecture']  # e.g. ['offset1-model', 'resample1-model']
        )
    ]).to(kwargs['device'])

    for m in model:
        m.to(kwargs['device'])
    for n, dataset in enumerate(data_loader.dataset.iter_sessions()):
        dataset.configure_for(model[n])

    criterion = (
        cebra.models.InfoMSE(temperature=kwargs['temperature'])
        if kwargs['distance'] == 'euclidean'
        else cebra.models.InfoNCE(temperature=kwargs['temperature'])
    )
    optimizer = torch.optim.Adam(
        itertools.chain(model.parameters(), criterion.parameters()),
        lr=kwargs['learning_rate']
    )
    return cebra.solver.MultiSessionSolver(
        model=model, criterion=criterion,
        optimizer=optimizer, tqdm_on=kwargs['verbose']
    )


@torch.no_grad()
def get_emissions(model, dataset):
    """
    Run the trained encoder forward on a dataset to get the embedding (no gradients).
    'Emissions' = encoder output = latent embedding vectors.
    Returns a NumPy array of shape (T, output_dimension).
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    dataset.configure_for(model)
    return model(dataset[torch.arange(len(dataset))].to(device)).cpu().numpy()

def compute_emissions_single(solver, dataset):
    """Get embedding from a SingleSessionSolver."""
    return get_emissions(solver.model, dataset)

def compute_emissions_multi(solver, dataset):
    """Get embeddings from a MultiSessionSolver — returns a dict {session_idx: embedding}."""
    return {
        i: get_emissions(model, session)
        for i, (model, session) in enumerate(
            zip(solver.model, dataset.iter_sessions())
        )
    }


def allen_frame_id_decode(train_fs, train_labels, test_fs, test_labels,
                          modality='neuropixel', decoder='knn'):
    """
    Decode movie frame IDs from CEBRA embeddings.  [Paper: Methods > 'Decoding']

    The paper evaluates decoding accuracy as:
       '% of test frames predicted within ±1 frame (30 ms window)'

    Pipeline:
    1. FRAME AVERAGING: Because Neuropixels runs at 120 Hz but the movie is at
       30 Hz, there are 4 time bins per frame. Average them → 1 vector per frame.
       Ca data is already at 30 Hz so FACTOR=1 (no averaging needed).

    2. HYPERPARAMETER SELECTION: Use repeat 9 (held-out validation) to pick the
       best k for kNN (from k=1,4,9,25,100) or smoothing for Naive Bayes.
       This prevents test-set leakage during model selection.

    3. FINAL TEST: Retrain on all 9 repeats with the best param, test on repeat 10.
       Report % predictions within ±1 frame of the true frame ID.

    Baselines compared in the paper:
       - kNN on raw neural activity (no CEBRA)
       - Naive Bayes on raw neural activity
       - CEBRA-NP (single session)
       - Joint CEBRA (Ca + NP together)
    """
    # FACTOR: how many time bins correspond to one 30Hz movie frame
    FACTOR     = 4 if modality == 'neuropixel' else 1
    time_window = 1  # accuracy window = ±1 frame

    def feature_for_one_frame(feature):
        """Average every FACTOR bins → one embedding vector per movie frame."""
        if isinstance(feature, torch.Tensor):
            feature = feature.cpu().numpy()
        return feature.reshape(-1, FACTOR, feature.shape[-1]).mean(axis=1)

    train_fs = feature_for_one_frame(train_fs)
    test_fs  = feature_for_one_frame(test_fs)

    if train_fs is None or test_fs is None:
        return [None], [None], None

    # Candidate hyperparameters: k in {1, 4, 9, 25, 100} for kNN
    params = (np.power(np.linspace(1, 10, 5, dtype=int), 2)
              if decoder == 'knn' else np.logspace(-9, 3, 5))
    errs = []

    for n in params:
        # Train on repeats 1-8, validate on repeat 9
        clf = (KNeighborsClassifier(n_neighbors=n, metric='cosine')
               if decoder == 'knn' else GaussianNB(var_smoothing=n))
        valid_idx = int(len(train_fs) / 9 * 8)  # 8/9 of training data
        clf.fit(train_fs[:valid_idx], train_labels[:valid_idx])
        pred = clf.predict(train_fs[valid_idx:])
        errs.append(abs(train_labels[valid_idx:] - pred).sum())

    # Retrain with best k on all 9 repeats, test on held-out repeat 10
    best_n = params[np.argmin(errs)]
    test_clf = (KNeighborsClassifier(n_neighbors=best_n, metric='cosine')
                if decoder == 'knn' else GaussianNB(var_smoothing=best_n))
    test_clf.fit(train_fs, train_labels)
    pred         = test_clf.predict(test_fs)
    frame_errors = pred - test_labels

    # Accuracy = fraction of predictions within ±1 frame (30 ms)
    quantized_acc = (abs(frame_errors) < (time_window * 30)).sum() / len(frame_errors) * 100
    return pred, frame_errors, quantized_acc


def consistency(feature1, feature2):
    """
    Measure linear consistency between two embedding spaces.  [Paper: Fig. 5]

    The paper defines consistency as the R² of a linear regression between two
    embedding spaces. This is used to quantify whether two independently trained
    CEBRA models (e.g., on Ca vs Neuropixels, or on two different brain areas)
    produce geometrically similar latent spaces.

    High R² (~0.9) = the two embeddings are linearly related → consistent structure.
    Low R² (~0.6)  = the embeddings differ → different neural representations.

    Bidirectional: fit from A→B and from B→A, report both.
    If Neuropixels (32400 = 900 frames × 4 bins × 9 repeats), downsample to
    frame-level first so Ca and NP embeddings have the same number of rows.
    """
    if len(feature1) == 32400:  # NP at 120 Hz: downsample to 30 Hz frame level
        feature1 = feature1.reshape(-1, 4, feature1.shape[-1]).mean(axis=1)
    if len(feature2) == 32400:
        feature2 = feature2.reshape(-1, 4, feature2.shape[-1]).mean(axis=1)

    def _linear_fit(a, b):
        lin = LinearRegression()
        lin.fit(a, b)
        return lin.score(a, b)  # Returns R²

    return _linear_fit(feature1, feature2), _linear_fit(feature2, feature1)


print('✅ All helper functions defined.')


### Step 1.4 — Train CEBRA-Behavior Models  [Paper: Fig. 4]

The paper trains **two embedding sizes**:
- **8D** → scatter plot visualizations (Fig. 4c,d,f,g)
- **128D** → decoding accuracy (Fig. 4 bar chart)

We use **seed=333** (one of the paper's 5 seeds). Results will be within ~3–5% of paper values.  
To fully replicate the averaged bars: change `seed` to each of `[111, 222, 333, 444, 555]`,
collect each run's accuracy, and average.


In [ ]:
# =============================================================================
# STEP 1.4 — TRAIN CEBRA-BEHAVIOR MODELS  [Paper: Fig. 4]
# =============================================================================
# ARCHITECTURES (paper Table S1):
#   Ca (30 Hz)    → 'offset1-model'   : conv net with 1-bin context window
#   NP (120 Hz)   → 'resample1-model' : resampling conv net (handles 120→30 Hz mismatch)
# Joint: one encoder per modality, both trained under the same InfoNCE loss against
#   shared DINO labels → consistent embedding space across Ca and NP.
#
# HYPERPARAMETERS (all match paper exactly):
#   num_steps=10000, batch_size=512, time_offset=1, conditional='time_delta'
#   distance='cosine', temperature=1.0, learning_rate=3e-4, num_hidden_units=128
#
# conditional='time_delta': positive pairs are matched by DINO feature similarity,
#   not just temporal adjacency — this is what makes CEBRA behavior-guided.

train_steps = 10000
seed        = 333  # One of the paper's 5 seeds: [111, 222, 333, 444, 555]
cortex      = 'VISp'
num_neurons = 800

print('Loading datasets (downloads ~200MB on first run)...')
ca_train    = cebra.datasets.init(f'allen-movie-one-ca-{cortex}-{num_neurons}-train-10-{seed}')
np_train    = cebra.datasets.init(f'allen-movie-one-neuropixel-{cortex}-{num_neurons}-train-10-{seed}')
joint_train = cebra.datasets.init(f'allen-movie-one-ca-neuropixel-{cortex}-{num_neurons}-train-10-{seed}')
np_test     = cebra.datasets.init(f'allen-movie-one-neuropixel-{cortex}-{num_neurons}-test-10-{seed}')
joint_test  = cebra.datasets.init(f'allen-movie-one-ca-neuropixel-{cortex}-{num_neurons}-test-10-{seed}')
print('Datasets loaded.')


def make_loaders(ca_ds, np_ds, joint_ds):
    """Data loaders must be re-created after each .fit() call (they are consumed)."""
    return (
        cebra.data.ContinuousDataLoader(ca_ds,    num_steps=train_steps, batch_size=512, conditional='time_delta', time_offset=1),
        cebra.data.ContinuousDataLoader(np_ds,    num_steps=train_steps, batch_size=512, conditional='time_delta', time_offset=1),
        cebra.data.ContinuousMultiSessionDataLoader(joint_ds, num_steps=train_steps, batch_size=512, conditional='time_delta', time_offset=1),
    )


# ── MODEL A: 8D — scatter plot visualization (Fig. 4c,d,f,g) ─────────────────
print('\nTraining 8D models (for scatter plots, Fig. 4c,d,f,g)...')
ca_l, np_l, joint_l = make_loaders(ca_train, np_train, joint_train)

m_ca_8d = single_session_solver(ca_l, model_architecture='offset1-model',
    distance='cosine', num_hidden_units=128, output_dimension=8,
    verbose=False, device=DEVICE, temperature=1, learning_rate=3e-4)
m_ca_8d.fit(ca_l)
emb_ca_8d = compute_emissions_single(m_ca_8d, ca_train)   # (8100, 8)

m_np_8d = single_session_solver(np_l, model_architecture='resample1-model',
    distance='cosine', num_hidden_units=128, output_dimension=8,
    verbose=False, device=DEVICE, temperature=1, learning_rate=3e-4)
m_np_8d.fit(np_l)
emb_np_8d = compute_emissions_single(m_np_8d, np_train)   # (32400, 8)

ca_l, np_l, joint_l = make_loaders(ca_train, np_train, joint_train)
m_joint_8d = multi_session_solver(joint_l, model_architecture=['offset1-model', 'resample1-model'],
    distance='cosine', num_hidden_units=128, output_dimension=8,
    verbose=False, device=DEVICE, temperature=1, learning_rate=3e-4)
m_joint_8d.fit(joint_l)
emb_joint_8d = compute_emissions_multi(m_joint_8d, joint_train)  # {0: Ca, 1: NP}
print(f'  Ca 8D: {emb_ca_8d.shape}  NP 8D: {emb_np_8d.shape}  Joint Ca: {emb_joint_8d[0].shape}  Joint NP: {emb_joint_8d[1].shape}')


# ── MODEL B: 128D — decoding accuracy (Fig. 4 bar chart) ─────────────────────
# Higher dimensionality → richer contrastive representation → better kNN accuracy.
print('\nTraining 128D models (for decoding bar chart, Fig. 4)...')
ca_l, np_l, joint_l = make_loaders(ca_train, np_train, joint_train)

m_np_128d = single_session_solver(np_l, model_architecture='resample1-model',
    distance='cosine', num_hidden_units=128, output_dimension=128,
    verbose=True, device=DEVICE, temperature=1, learning_rate=3e-4)
m_np_128d.fit(np_l)
emb_np_128d = compute_emissions_single(m_np_128d, np_train)  # (32400, 128)

ca_l, np_l, joint_l = make_loaders(ca_train, np_train, joint_train)
m_joint_128d = multi_session_solver(joint_l, model_architecture=['offset1-model', 'resample1-model'],
    distance='cosine', num_hidden_units=128, output_dimension=128,
    verbose=True, device=DEVICE, temperature=1, learning_rate=3e-4)
m_joint_128d.fit(joint_l)
emb_joint_128d = compute_emissions_multi(m_joint_128d, joint_train)  # {0: Ca, 1: NP}
print(f'  NP 128D: {emb_np_128d.shape}  Joint NP 128D: {emb_joint_128d[1].shape}')

print('\nAll models trained.')


In [ ]:
# =============================================================================
# VISUALIZE 8D EMBEDDINGS  [Paper: Fig. 4c,d,f,g]
# =============================================================================
# First 2 of 8 latent dimensions shown. Color = frame index in movie (dark→light).
# Expected: ring structure where all 9 training repeats overlap.
# Jointly trained Ca and NP should show similar geometry — cross-modality consistency.

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
panels = [
    (axes[0,0], emb_ca_8d,        np.tile(np.arange(900), 9),            f'Ca — single session (8D) — Fig. 4c'),
    (axes[0,1], emb_np_8d,        np.tile(np.repeat(np.arange(900),4),9),f'Neuropixels — single session (8D) — Fig. 4d'),
    (axes[1,0], emb_joint_8d[0],  np.tile(np.arange(900), 9),            f'Ca — jointly trained (8D) — Fig. 4f'),
    (axes[1,1], emb_joint_8d[1],  np.tile(np.repeat(np.arange(900),4),9),f'Neuropixels — jointly trained (8D) — Fig. 4g'),
]
for ax, emb, colors, title in panels:
    ax.scatter(emb[:, 0], emb[:, 1], c=colors, cmap='magma', s=1)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle(f'CEBRA-Behavior 8D Embeddings — {cortex}, {num_neurons} neurons, seed={seed}\n'
             'Color = frame position in movie. Ring structure = movie stimulus encoded.',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig4_embeddings_8D.png', dpi=150, bbox_inches='tight')
plt.show()


### Step 1.6 — Decode Movie Frame IDs  [Paper: Fig. 4 bar chart]

Metric: **% of held-out test frames (repeat 10) predicted within ±1 frame (±33 ms)**.  
Decoder trained on repeats 1–8, hyperparameter (k) selected on repeat 9, tested on repeat 10.

| Condition | Expected |
|-----------|----------|
| kNN on raw NP | ~55% |
| Naive Bayes on raw NP | ~68% |
| CEBRA-NP (128D) → kNN | ~82% |
| Joint CEBRA (Ca+NP, 128D) → kNN | ~89% |


In [ ]:
# =============================================================================
# STEP 1.6 — FRAME DECODING  [Paper: Fig. 4 bar chart, Methods > 'Decoding']
# =============================================================================
# The allen_frame_id_decode() function (defined in Step 1.3) handles:
#   1. Frame averaging (4 NP bins → 1 frame for the 120→30 Hz alignment)
#   2. k selection via repeat-9 validation
#   3. Test on repeat 10 only

train_labels = np.tile(np.arange(900), 9)  # frame IDs, 9 training repeats
test_labels  = np.arange(900)               # frame IDs, test repeat 10

emb_np_test    = compute_emissions_single(m_np_128d, np_test)
emb_joint_test = compute_emissions_multi(m_joint_128d, joint_test)

print('Decoding from held-out test repeat...')
_, _, acc_knn   = allen_frame_id_decode(np_train.neural, train_labels, np_test.neural,       test_labels, modality='neuropixel', decoder='knn')
_, _, acc_bayes = allen_frame_id_decode(np_train.neural, train_labels, np_test.neural,       test_labels, modality='neuropixel', decoder='bayes')
_, _, acc_cebra = allen_frame_id_decode(emb_np_128d,     train_labels, emb_np_test,          test_labels, modality='neuropixel', decoder='knn')
_, _, acc_joint = allen_frame_id_decode(emb_joint_128d[1], train_labels, emb_joint_test[1], test_labels, modality='neuropixel', decoder='knn')

print(f'  kNN baseline   : {acc_knn:.1f}%   [paper ~55%]')
print(f'  Naive Bayes    : {acc_bayes:.1f}%   [paper ~68%]')
print(f'  CEBRA-NP 128D  : {acc_cebra:.1f}%   [paper ~82%]')
print(f'  Joint CEBRA    : {acc_joint:.1f}%   [paper ~89%]')
print('\nNote: paper values are means across 5 seeds; ±3-5% per seed is normal.')


In [ ]:
# ── Bar chart — Paper Fig. 4 ──────────────────────────────────────────────────
# Tick marks show the approximate paper values (mean of 5 seeds) for reference.

labels = ['kNN\n(raw NP)', 'Naive Bayes\n(raw NP)', 'CEBRA-NP\n(128D)', 'Joint CEBRA\n(Ca+NP, 128D)']
values = [acc_knn, acc_bayes, acc_cebra, acc_joint]
paper  = [55, 68, 82, 89]
colors = ['#AAAAAA', '#AAAAAA', '#2E75B6', '#1F4E79']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
ax.scatter(labels, paper, marker='_', s=500, color='black', linewidths=2.5,
           zorder=5, label='Paper value (5-seed mean)')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('Frame decoding accuracy (% within ±1 frame)', fontsize=11)
ax.set_title(f'Fig. 4 — Movie Frame Decoding from {cortex}\n{num_neurons} neurons, seed={seed}', fontsize=12)
ax.set_ylim(0, 105)
ax.axhline(3/900*100, color='red', linestyle='--', alpha=0.5, label='Chance (~0.33%)')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig4_decoding_bar_chart.png', dpi=150, bbox_inches='tight')
plt.show()


### Step 1.7 — Intra- vs. Inter-Area Consistency  [Paper: Fig. 5]

Measures whether CEBRA embeddings are **consistent across recording modalities** within the same area,
and whether this consistency is higher within an area than between areas.

**Metric:** R² of linear regression between two embedding spaces (bidirectional).  
- **Intra-area** (VISp Ca ↔ VISp NP): expected ~0.90  
- **Inter-area** (VISp ↔ VISrl): expected ~0.60  
- Paper: one-sided Welch t-test T=4.55, p=0.00019

**Disjoint neurons:** `disjoint-0-400` datasets use the first 400 neurons from each area —
non-overlapping subsets rule out trivial consistency from shared neurons.


In [ ]:
# =============================================================================
# STEP 1.7 — INTRA vs INTER-AREA CONSISTENCY  [Paper: Fig. 5]
# =============================================================================
# SETTINGS (smaller than decoding — consistency is robust to model size):
#   model_architecture: ['offset10-model', 'resample-model']  (10-bin context for Ca)
#   output_dimension  : 32    (paper uses 32D for consistency experiments)
#   num_steps         : 1000  (shorter; consistency converges quickly)
#   time_offset       : 10    (larger temporal context for this training mode)

c1, c2 = 'VISp', 'VISrl'

print(f'Loading disjoint datasets ({c1}, {c2})...')
data_c1 = cebra.datasets.init(f'allen-movie-one-ca-neuropixel-{c1}-disjoint-0-400-train-10-{seed}')
data_c2 = cebra.datasets.init(f'allen-movie-one-ca-neuropixel-{c2}-disjoint-0-400-train-10-{seed}')

def make_cons_loader(ds):
    return cebra.data.ContinuousMultiSessionDataLoader(
        ds, num_steps=1000, batch_size=512, conditional='time_delta', time_offset=10)

loader_c1 = make_cons_loader(data_c1)
model_c1 = multi_session_solver(loader_c1,
    model_architecture=['offset10-model', 'resample-model'],
    distance='cosine', num_hidden_units=32, output_dimension=32,
    verbose=True, device=DEVICE, temperature=1, learning_rate=3e-4)

loader_c2 = make_cons_loader(data_c2)
model_c2 = multi_session_solver(loader_c2,
    model_architecture=['offset10-model', 'resample-model'],
    distance='cosine', num_hidden_units=32, output_dimension=32,
    verbose=True, device=DEVICE, temperature=1, learning_rate=3e-4)

print(f'\nTraining {c1} consistency model...')
model_c1.fit(loader_c1)
emb_c1 = compute_emissions_multi(model_c1, data_c1)  # {0: Ca, 1: NP}

print(f'Training {c2} consistency model...')
model_c2.fit(loader_c2)
emb_c2 = compute_emissions_multi(model_c2, data_c2)

# Intra: Ca vs NP within same area
intra_r2 = consistency(emb_c1[0], emb_c1[1])

# Inter: all 4 pairings across areas
inter_r2 = []
for ea in emb_c1.values():
    for eb in emb_c2.values():
        inter_r2.extend(consistency(ea, eb))

print(f'\n── Consistency (linear R²) ─────────────────────────────────')
print(f'  Intra-area ({c1}, Ca↔NP) : {np.mean(intra_r2):.3f}   [paper ~0.90]')
print(f'  Inter-area ({c1}↔{c2})  : {np.mean(inter_r2):.3f}   [paper ~0.60]')
print(f'  Difference               : {np.mean(intra_r2)-np.mean(inter_r2):.3f}')


In [ ]:
# ── Consistency plot — Paper Fig. 5 ───────────────────────────────────────────

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter([0]*len(intra_r2), list(intra_r2), color='steelblue', s=70, zorder=3, label='Intra-area')
ax.scatter([1]*len(inter_r2), list(inter_r2), color='gray', s=40, zorder=3, alpha=0.7, label='Inter-area')
ax.plot([0, 1], [np.mean(intra_r2), np.mean(inter_r2)], 'ko-', lw=2, ms=10, zorder=4, label='Mean')
ax.set_xticks([0, 1])
ax.set_xticklabels([f'Intra-area\n({c1} Ca↔NP)', f'Inter-area\n({c1}↔{c2})'], fontsize=11)
ax.set_ylabel('Linear consistency (R²)', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title(f'Fig. 5 — Intra vs. Inter-Area Consistency\n32D joint CEBRA-Behavior models', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig5_consistency.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Results Summary

| Figure | Metric | This run | Paper (5-seed mean) |
|--------|--------|----------|---------------------|
| Fig. 4 | kNN baseline | — | ~55% |
| Fig. 4 | Naive Bayes | — | ~68% |
| Fig. 4 | CEBRA-NP (128D) | — | ~82% |
| Fig. 4 | Joint CEBRA (128D) | — | ~89% |
| Fig. 5 | Intra-area R² | — | ~0.90 |
| Fig. 5 | Inter-area R² | — | ~0.60 |

Fill in the dashes after running all steps. ±3–5% deviation from paper values is expected with one seed.

**Saved figures:** `fig4_embeddings_8D.png`, `fig4_decoding_bar_chart.png`, `fig5_consistency.png`


In [ ]:
# Run after all steps complete
print('=' * 55)
print('RESULTS vs. PAPER')
print('=' * 55)
print(f'Settings: {cortex}, {num_neurons} neurons, seed={seed}')
print()
print('Fig. 4 — Frame Decoding (% within +-1 frame):')
print(f'  kNN baseline   : {acc_knn:.1f}%   [paper ~55%]')
print(f'  Naive Bayes    : {acc_bayes:.1f}%   [paper ~68%]')
print(f'  CEBRA-NP (128D): {acc_cebra:.1f}%   [paper ~82%]')
print(f'  Joint CEBRA    : {acc_joint:.1f}%   [paper ~89%]')
print()
print('Fig. 5 — Embedding Consistency (linear R2):')
print(f'  Intra-area ({c1}) : {np.mean(intra_r2):.3f}   [paper ~0.90]')
print(f'  Inter-area ({c1}/{c2}): {np.mean(inter_r2):.3f}   [paper ~0.60]')
